# Notebook kerja Teman B (experiments, train, sample)
Sel berlabel **SALINAN SEMENTARA** cuma dipakai buat uji di notebook ini dan dibuang waktu ketua merakit `MiniDDPM.ipynb`. Bagian Teman B mulai dari "8. Experiments".

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
import math
import random
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

In [ ]:
# Import Teman B (bagian 1). Saat merakit, ketua menyatukannya ke sel import utama.
import copy
import json
import time

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Root repo: folder kerja saat ini, atau folder induknya kalau dijalankan dari notebooks/
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
config = SimpleNamespace(
    # Reproducibility
    SEED=42,  # dipakai set_seed(); satu seed untuk semua anggota
    # Training (baseline)
    LEARNING_RATE=2e-4,  # baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # SEMENTARA, BELUM FINAL: menunggu uji memori di GTX 1650 Ti (4GB).
    # Jangan dipakai untuk eksperimen tuning sebelum nilai final ditetapkan.
    BATCH_SIZE=64,
    EPOCHS=30,
    NUM_WORKERS=0,  # DataLoader tanpa proses worker (aman di macOS dan Windows)
    # Diffusion (baseline)
    T=1000,  # jumlah timestep baseline; boleh ditimpa konfigurasi di EXPERIMENTS
    # Acuan untuk T=1000; Diffusion menskalakan beta dengan BETA_REFERENCE_T / T
    BETA_START=1e-4,
    BETA_END=0.02,
    BETA_REFERENCE_T=1000,
    # Model
    IMAGE_SIZE=28,  # sisi gambar MNIST; channel ada di MODEL_PARAMS
    # Dipakai sebagai UNet(**config.MODEL_PARAMS) dan disimpan di checkpoint.
    # channel_mults (1, 2, 2) = dua downsampling: 28 -> 14 -> 7
    MODEL_PARAMS={
        "in_channels": 1,
        "base_channels": 64,
        "channel_mults": (1, 2, 2),
        "time_emb_dim": 128,
    },
)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Sampling selama training
config.SAMPLE_EVERY_N_EPOCHS = 5  # grid sampel disimpan tiap N epoch
config.NUM_SAMPLE_IMAGES = 16     # jumlah gambar per grid

# Smoke test (uji alur cepat, bukan untuk hasil)
config.SMOKE_TEST = False         # True = subset kecil dan epoch pendek
config.SMOKE_SUBSET_SIZE = 512    # jumlah gambar pada mode smoke test
config.SMOKE_EPOCHS = 2           # jumlah epoch pada mode smoke test
config.SMOKE_SAMPLE_EVERY = 1     # interval grid smoke test (harus <= SMOKE_EPOCHS)

# Path: semuanya diturunkan dari BASE_DIR (sel sebelumnya)
config.BASE_DIR = BASE_DIR
config.DATA_DIR = BASE_DIR / "data"               # unduhan MNIST (tidak di-commit)
config.CHECKPOINT_DIR = BASE_DIR / "checkpoints"  # checkpoint (tidak di-commit)
config.RESULTS_DIR = BASE_DIR / "results"         # hasil (di-commit)
config.LOSS_CURVES_DIR = config.RESULTS_DIR / "loss_curves"
config.SAMPLES_DIR = config.RESULTS_DIR / "samples"

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Override khusus laptop CPU Gita: smoke test aktif dan sampling dibuat murah.
config.SMOKE_TEST = True
config.NUM_SAMPLE_IMAGES = 4
config.T = 50

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def set_seed(seed):
    """Seed random, numpy, torch. Tanpa mode deterministik penuh agar training tidak melambat."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # sekaligus menyemai CUDA dan MPS

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def get_device():
    """Device pertama yang tersedia: cuda, mps, cpu. Tanpa print."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Pengganti get_dataloader Teman A: data acak, bukan MNIST, dan hanya untuk smoke test.
# Sama seperti kontrak versi asli: SMOKE_TEST dan SMOKE_SUBSET_SIZE dibaca dari argumen cfg, bukan config global.
from torch.utils.data import DataLoader


def get_dataloader(cfg):
    if not cfg.SMOKE_TEST:
        raise RuntimeError("Pengganti ini hanya untuk smoke test; pakai get_dataloader asli Teman A")
    shape = (cfg.SMOKE_SUBSET_SIZE, cfg.MODEL_PARAMS["in_channels"], cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)
    images = torch.rand(shape) * 2 - 1  # float dalam [-1, 1], tanpa label
    # Tensor dipakai langsung sebagai dataset, jadi tiap batch = satu tensor (B, 1, 28, 28)
    return DataLoader(images, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Pengganti save_image_grid Teman A (versi minimal): simpan lalu tampilkan.
def save_image_grid(images, path):
    images = (images.detach().cpu().clamp(-1, 1) + 1) / 2  # [-1, 1] -> [0, 1]
    n = images.shape[0]
    cols = math.ceil(math.sqrt(n))
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols, rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
# Pengganti plot_loss Teman A (versi minimal): series = {label: [loss per epoch]}.
def plot_loss(series, path):
    fig, ax = plt.subplots(figsize=(6, 4))
    for label, losses in series.items():
        ax.plot(range(1, len(losses) + 1), losses, marker="o", label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE loss")
    ax.legend()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def _group_norm(channels):
    # Jumlah grup terbesar dari (8, 4, 2, 1) yang habis membagi jumlah channel
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return nn.GroupNorm(groups, channels)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class SinusoidalTimeEmbedding(nn.Module):
    """t (B,) bertipe long -> vektor (B, dim) berisi sin dan cos berbagai frekuensi."""

    def __init__(self, dim):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("time_emb_dim harus genap")
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        # Frekuensi turun geometris dari 1 ke 1/10000
        exponent = -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        freqs = torch.exp(exponent)                      # (half,)
        args = t.float()[:, None] * freqs[None, :]       # (B, half)
        return torch.cat([args.sin(), args.cos()], dim=1)  # (B, dim)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class ResBlock(nn.Module):
    """Dua konvolusi 3x3; embedding waktu ditambahkan di antaranya."""

    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.norm1 = _group_norm(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch))
        self.norm2 = _group_norm(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act = nn.SiLU()
        # Jalan pintas residual; 1x1 conv bila jumlah channel berubah
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]  # (B,C) -> (B,C,1,1), disiarkan ke semua piksel
        h = self.conv2(self.act(self.norm2(h)))
        return h + self.shortcut(x)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)  # 28->14, 14->7

    def forward(self, x):
        return self.conv(x)

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")  # 7->14, 14->28
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(self.up(x))

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class UNet(nn.Module):
    def __init__(self, in_channels, base_channels, channel_mults, time_emb_dim):
        super().__init__()
        n = len(channel_mults)
        if not 1 <= n <= 3:
            raise ValueError("channel_mults harus berisi 1 sampai 3 level (maks. dua downsampling)")
        chs = [base_channels * m for m in channel_mults]
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.init_conv = nn.Conv2d(in_channels, chs[0], 3, padding=1)
        self.down_blocks = nn.ModuleList(
            [ResBlock(chs[max(i - 1, 0)], chs[i], time_emb_dim) for i in range(n)]
        )
        self.downsamples = nn.ModuleList([Downsample(chs[i]) for i in range(n - 1)])
        self.mid1 = ResBlock(chs[-1], chs[-1], time_emb_dim)
        self.mid2 = ResBlock(chs[-1], chs[-1], time_emb_dim)

        # Jalur naik dari level terdalam: input = hasil level bawah + skip
        self.up_blocks = nn.ModuleList()
        for k in range(n):
            i = n - 1 - k
            self.up_blocks.append(
                ResBlock(chs[i] + chs[min(i + 1, n - 1)], chs[i], time_emb_dim)
            )
        self.upsamples = nn.ModuleList([Upsample(chs[n - 1 - k]) for k in range(n - 1)])
        self.out_norm = _group_norm(chs[0])
        self.out_conv = nn.Conv2d(chs[0], in_channels, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.init_conv(x)
        skips = []
        for i, block in enumerate(self.down_blocks):
            h = block(h, t_emb)
            skips.append(h)
            if i < len(self.downsamples):
                h = self.downsamples[i](h)
        h = self.mid2(self.mid1(h, t_emb), t_emb)

        for k, block in enumerate(self.up_blocks):
            h = torch.cat([h, skips.pop()], dim=1)  # skip dari level seukuran
            h = block(h, t_emb)
            if k < len(self.upsamples):
                h = self.upsamples[k](h)
        return self.out_conv(self.act(self.out_norm(h)))

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
def _extract(values, t, x):
    """Ambil values[t] lalu ubah ke (B,1,1,1) agar bisa dikalikan dengan x (B,C,H,W)."""
    out = values.to(x.device)[t.to(x.device)]
    return out.reshape(t.shape[0], *([1] * (x.dim() - 1)))

In [ ]:
# SALINAN SEMENTARA, jangan dimasukkan ke notebook final
class Diffusion:
    def __init__(self, T, device=None,
                 beta_start=config.BETA_START, beta_end=config.BETA_END):
        # Beta diskalakan BETA_REFERENCE_T / T supaya total noise tetap sama saat T berubah
        scale = config.BETA_REFERENCE_T / T
        if beta_end * scale >= 1.0:
            raise ValueError(f"T={T} terlalu kecil: beta akhir jadi >= 1")
        self.T = T
        self.betas = torch.linspace(beta_start * scale, beta_end * scale, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bars = self.alpha_bars.sqrt()
        self.sqrt_one_minus_alpha_bars = (1.0 - self.alpha_bars).sqrt()
        self.sqrt_recip_alphas = (1.0 / self.alphas).sqrt()
        self.sqrt_betas = self.betas.sqrt()  # simpangan baku noise tambahan saat sampling

    def add_noise(self, x0, t, noise):
        # Forward process: x_t = sqrt(a_bar_t) * x0 + sqrt(1 - a_bar_t) * noise
        return (_extract(self.sqrt_alpha_bars, t, x0) * x0
                + _extract(self.sqrt_one_minus_alpha_bars, t, x0) * noise)

    def sample_timesteps(self, batch_size, device):
        return torch.randint(0, self.T, (batch_size,), device=device)  # t acak long, 0..T-1

    @torch.no_grad()
    def sample(self, model, num_images, device):
        # Reverse process; mode eval/train model diatur pemanggil (train)
        size = config.IMAGE_SIZE
        shape = (num_images, config.MODEL_PARAMS["in_channels"], size, size)
        x = torch.randn(shape, device=device)
        for i in reversed(range(self.T)):
            t = torch.full((num_images,), i, device=device, dtype=torch.long)
            pred_noise = model(x, t)
            # Buang noise yang diprediksi -> rata-rata estimasi x_{t-1}
            coef = _extract(self.betas, t, x) / _extract(self.sqrt_one_minus_alpha_bars, t, x)
            mean = _extract(self.sqrt_recip_alphas, t, x) * (x - coef * pred_noise)
            # Noise baru tiap langkah, kecuali langkah terakhir (t=0)
            if i > 0:
                x = mean + _extract(self.sqrt_betas, t, x) * torch.randn_like(x)
            else:
                x = mean
        return x.clamp(-1.0, 1.0)

## 8. Experiments (Teman B)
Tiap eksperimen cuma boleh ngubah satu hal, learning rate atau T, sisanya ikut baseline di `config`. Jadi kalau hasilnya beda, kita tahu penyebabnya dari variabel yang mana.

In [ ]:
EXPERIMENTS = {
    "baseline": {},                      # semua nilai dari config
    "lr_1e-4": {"LEARNING_RATE": 1e-4},  # learning rate lebih kecil dari baseline
    "lr_5e-4": {"LEARNING_RATE": 5e-4},  # learning rate lebih besar dari baseline
    "T_250": {"T": 250},                 # timestep lebih sedikit
    "T_500": {"T": 500},
}

`_resolve_run` menggabungkan baseline dengan timpaan eksperimen ke dalam **salinan** config, jadi `config` aslinya nggak pernah diubah. Di sini juga semua nama file ditentukan, dan akhiran `_smoke` ditambahkan kalau mode smoke supaya hasil uji nggak menimpa hasil asli.

In [ ]:
def _resolve_run(name, smoke=None):
    """Nilai efektif satu konfigurasi sebagai salinan lokal. smoke=None ikut config.SMOKE_TEST."""
    if name not in EXPERIMENTS:
        raise KeyError(f"Konfigurasi '{name}' tidak ada. Pilihan: {list(EXPERIMENTS)}")
    extra = set(EXPERIMENTS[name]) - {"LEARNING_RATE", "T"}
    if extra:
        raise ValueError(f"{name}: hanya LEARNING_RATE dan T yang boleh ditimpa, bukan {extra}")
    cfg = SimpleNamespace(**copy.deepcopy(vars(config)))  # salinan; config asli tidak disentuh
    for key, value in EXPERIMENTS[name].items():
        setattr(cfg, key, value)
    cfg.SMOKE_TEST = config.SMOKE_TEST if smoke is None else smoke
    suffix = "_smoke" if cfg.SMOKE_TEST else ""
    tag = name + suffix
    return SimpleNamespace(
        name=name, cfg=cfg, tag=tag, suffix=suffix,
        lr=cfg.LEARNING_RATE, T=cfg.T,
        epochs=cfg.SMOKE_EPOCHS if cfg.SMOKE_TEST else cfg.EPOCHS,
        sample_every=cfg.SMOKE_SAMPLE_EVERY if cfg.SMOKE_TEST else cfg.SAMPLE_EVERY_N_EPOCHS,
        checkpoint_path=cfg.CHECKPOINT_DIR / f"ddpm_{tag}.pt",
        loss_json=cfg.LOSS_CURVES_DIR / f"{tag}_loss.json",
        loss_png=cfg.LOSS_CURVES_DIR / f"{tag}_loss.png",
    )

In [ ]:
# Uji experiments
snapshot = copy.deepcopy(vars(config))
assert EXPERIMENTS["baseline"] == {}, "baseline tidak boleh punya timpaan"
for name, overrides in EXPERIMENTS.items():
    assert set(overrides) <= {"LEARNING_RATE", "T"}, f"{name}: hanya LR atau T"
    assert len(overrides) <= 1, f"{name}: maksimal satu variabel per eksperimen"
    run = _resolve_run(name)
    assert isinstance(run.T, int) and run.T > 20, f"{name}: T harus bilangan bulat > 20"
    assert run.lr > 0
    print(f"{name:<9} lr={run.lr:<7g} T={run.T:<4} epochs={run.epochs}  file={run.checkpoint_path.name}")

smoke_run = _resolve_run("baseline", smoke=True)
full_run = _resolve_run("baseline", smoke=False)
assert smoke_run.epochs == config.SMOKE_EPOCHS and smoke_run.tag == "baseline_smoke"
assert full_run.epochs == config.EPOCHS and full_run.tag == "baseline"
try:
    _resolve_run("tidak_ada")
    raise AssertionError("nama konfigurasi salah seharusnya ditolak")
except KeyError:
    print("nama salah ditolak OK")
assert vars(config) == snapshot, "config tidak boleh berubah"
print("experiments OK")

## 9. Train (Teman B)
Tiap iterasi: ambil batch gambar, pilih `t` acak, tambahkan noise, suruh model menebak noise itu, lalu hitung MSE antara noise asli dan tebakan. Parameter `smoke` di `train` cuma jalan kalau `get_dataloader` membaca `SMOKE_TEST` dan `SMOKE_SUBSET_SIZE` dari argumen `cfg`, bukan dari `config` global.

In [ ]:
def _train_one_epoch(model, loader, diffusion, optimizer, device):
    """Satu putaran data; mengembalikan rata-rata loss MSE per batch."""
    model.train()
    total = torch.zeros((), device=device)
    n_batches = 0
    for x0 in loader:  # tiap batch: satu tensor (B, 1, 28, 28) dalam [-1, 1]
        x0 = x0.to(device)
        t = diffusion.sample_timesteps(x0.shape[0], device)
        noise = torch.randn_like(x0)
        x_t = diffusion.add_noise(x0, t, noise)
        loss = nn.functional.mse_loss(model(x_t, t), noise)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += loss.detach()  # tetap di device, tanpa sinkronisasi tiap batch
        n_batches += 1
    return (total / n_batches).item()

Checkpoint cuma berisi tensor, angka, dan dict biasa, jadi aman dibuka dengan `torch.load(weights_only=True)`. Isinya cukup buat `sample` membangun ulang model, termasuk T, karena T yang salah bikin noise schedule-nya beda.

In [ ]:
def _save_checkpoint(path, model, optimizer, T, model_params, epoch):
    """Simpan lima isi checkpoint; tanpa objek config atau Path."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": model.state_dict(),
        "T": int(T),
        "MODEL_PARAMS": dict(model_params),
        "epoch": int(epoch),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)

In [ ]:
def _save_loss_record(run, losses, epoch_seconds):
    """Catatan loss per epoch ke JSON, supaya bagian 13 bisa membandingkan tanpa melatih ulang."""
    run.loss_json.parent.mkdir(parents=True, exist_ok=True)
    record = {
        "name": run.name,
        "smoke": run.cfg.SMOKE_TEST,
        "learning_rate": run.lr,
        "T": run.T,
        "losses": losses,
        "epoch_seconds": epoch_seconds,  # waktu training per epoch, tanpa sampling
    }
    run.loss_json.write_text(json.dumps(record, indent=2))

In [ ]:
def _save_training_grid(model, diffusion, run, epoch, device):
    """Grid sampel di tengah training; mengembalikan lama sampling dalam detik."""
    model.eval()
    start = time.perf_counter()
    images = diffusion.sample(model, run.cfg.NUM_SAMPLE_IMAGES, device)  # sudah tanpa gradien
    seconds = time.perf_counter() - start
    model.train()
    save_image_grid(images, run.cfg.SAMPLES_DIR / f"{run.tag}_epoch{epoch:03d}.png")
    return seconds

In [ ]:
def _log_epoch(run, epoch, loss, train_s, sample_s, elapsed_s):
    """Cetak progres satu epoch plus perkiraan sisa waktu dari rata-rata epoch sejauh ini."""
    eta_min = elapsed_s / epoch * (run.epochs - epoch) / 60
    msg = f"[{run.tag}] epoch {epoch}/{run.epochs}  loss={loss:.4f}  train={train_s:.1f}s"
    if sample_s:
        msg += f"  sampling={sample_s:.1f}s"
    print(f"{msg}  sisa≈{eta_min:.1f} menit")

In [ ]:
def train(name="baseline", smoke=None):
    """Latih satu konfigurasi EXPERIMENTS. smoke=None ikut config.SMOKE_TEST; True/False menimpanya
    lewat salinan lokal (butuh get_dataloader yang membaca SMOKE_* dari argumen cfg)."""
    run = _resolve_run(name, smoke)
    set_seed(config.SEED)  # inisialisasi dan urutan data sama untuk semua eksperimen
    device = get_device()
    loader = get_dataloader(run.cfg)
    model = UNet(**run.cfg.MODEL_PARAMS).to(device)
    diffusion = Diffusion(run.T, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=run.lr)
    print(f"[{run.tag}] device={device}  lr={run.lr:g}  T={run.T}  epochs={run.epochs}")

    losses, epoch_seconds, start_all = [], [], time.perf_counter()
    for epoch in range(1, run.epochs + 1):
        start = time.perf_counter()
        losses.append(_train_one_epoch(model, loader, diffusion, optimizer, device))
        epoch_seconds.append(time.perf_counter() - start)
        is_grid_epoch = epoch % run.sample_every == 0
        if is_grid_epoch or epoch == run.epochs:  # checkpoint berkala + akhir, satu file ditimpa
            _save_checkpoint(run.checkpoint_path, model, optimizer, run.T, run.cfg.MODEL_PARAMS, epoch)
        sample_s = _save_training_grid(model, diffusion, run, epoch, device) if is_grid_epoch else 0.0
        _save_loss_record(run, losses, epoch_seconds)
        _log_epoch(run, epoch, losses[-1], epoch_seconds[-1], sample_s, time.perf_counter() - start_all)

    plot_loss({run.name: losses}, run.loss_png)
    return run.checkpoint_path, losses

In [ ]:
# Uji train: selalu smoke, jadi tetap cepat walaupun notebook final pakai SMOKE_TEST=False
snapshot = copy.deepcopy(vars(config))
smoke_run = _resolve_run("baseline", smoke=True)
ckpt_smoke, losses_smoke = train("baseline", smoke=True)

assert ckpt_smoke == smoke_run.checkpoint_path and ckpt_smoke.exists()
assert len(losses_smoke) == smoke_run.epochs, "loss harus tercatat tiap epoch"
assert all(math.isfinite(v) for v in losses_smoke)
record = json.loads(smoke_run.loss_json.read_text())
assert record["losses"] == losses_smoke and len(record["epoch_seconds"]) == smoke_run.epochs
assert smoke_run.loss_png.exists()
first_grid = config.SAMPLES_DIR / f"{smoke_run.tag}_epoch{smoke_run.sample_every:03d}.png"
assert first_grid.exists(), "grid harus tersimpan minimal sekali"
assert vars(config) == snapshot, "config tidak boleh berubah setelah train"
print("train OK")

In [ ]:
# Uji checkpoint: aman untuk weights_only dan isinya cukup untuk membangun ulang model
ckpt = torch.load(ckpt_smoke, map_location=get_device(), weights_only=True)
assert set(ckpt) == {"model_state_dict", "T", "MODEL_PARAMS", "epoch", "optimizer_state_dict"}
assert ckpt["T"] == smoke_run.T and ckpt["epoch"] == smoke_run.epochs
assert ckpt["MODEL_PARAMS"] == config.MODEL_PARAMS
assert ckpt["optimizer_state_dict"]["param_groups"][0]["lr"] == smoke_run.lr
rebuilt = UNet(**ckpt["MODEL_PARAMS"])
rebuilt.load_state_dict(ckpt["model_state_dict"])  # error kalau bentuk bobot tidak cocok
print(f"checkpoint OK: T={ckpt['T']}  epoch={ckpt['epoch']}  file={ckpt_smoke.name}")

## 10. Sample (Teman B)
`sample(path_checkpoint)` membuka checkpoint, membangun ulang `UNet` dan `Diffusion` dari isinya (bukan dari `config`), lalu memanggil `Diffusion.sample` milik ketua. Jadi fungsi ini nggak punya logika reverse process sendiri.

In [ ]:
def sample(path_checkpoint):
    """Muat checkpoint, bangun ulang model dan noise schedule dari isinya, lalu buat grid sampel."""
    path_checkpoint = Path(path_checkpoint)
    device = get_device()
    ckpt = torch.load(path_checkpoint, map_location=device, weights_only=True)
    model = UNet(**ckpt["MODEL_PARAMS"]).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    diffusion = Diffusion(ckpt["T"], device)  # T dari checkpoint, bukan config
    set_seed(config.SEED)  # noise awal sama tiap dipanggil, biar grid antar-eksperimen adil
    images = diffusion.sample(model, config.NUM_SAMPLE_IMAGES, device)
    save_image_grid(images, config.SAMPLES_DIR / f"{path_checkpoint.stem}_final.png")
    print(f"sample: {path_checkpoint.name}  T={ckpt['T']}  epoch={ckpt['epoch']}  ->  {images.shape[0]} gambar")
    return images

In [ ]:
# Uji sample: checkpoint dari uji train bisa dimuat dan menghasilkan (N, 1, 28, 28) dalam [-1, 1]
images = sample(ckpt_smoke)
expected = (config.NUM_SAMPLE_IMAGES, config.MODEL_PARAMS["in_channels"], config.IMAGE_SIZE, config.IMAGE_SIZE)
assert images.shape == expected, f"shape salah: {tuple(images.shape)}"
assert torch.isfinite(images).all() and images.min() >= -1 and images.max() <= 1
assert (config.SAMPLES_DIR / f"{ckpt_smoke.stem}_final.png").exists()
print("sample OK")

In [ ]:
# Uji rebuild: checkpoint model mini -> sample harus membangun model dari checkpoint, bukan dari config
device = get_device()
mini_params = dict(config.MODEL_PARAMS, base_channels=config.MODEL_PARAMS["base_channels"] // 4)
mini_T = min(_resolve_run(n).T for n in EXPERIMENTS)
mini_model = UNet(**mini_params).to(device)
mini_opt = torch.optim.Adam(mini_model.parameters(), lr=config.LEARNING_RATE)
mini_path = config.CHECKPOINT_DIR / "ddpm_uji_rebuild_smoke.pt"
_save_checkpoint(mini_path, mini_model, mini_opt, mini_T, mini_params, epoch=0)

images = sample(mini_path)  # gagal load_state_dict kalau sample memakai config.MODEL_PARAMS
assert images.shape[0] == config.NUM_SAMPLE_IMAGES
mini_path.unlink()
(config.SAMPLES_DIR / f"{mini_path.stem}_final.png").unlink()
print(f"rebuild dari checkpoint OK (base_channels={mini_params['base_channels']}, T={mini_T})")

## 12. Eksekusi baseline (Teman B)
Latih konfigurasi `baseline`: grafik loss dan grid tiap N epoch langsung tampil dan tersimpan di `results/`. Setelah itu kita sampling ulang dari checkpoint untuk memastikan model bisa dipakai tanpa melatih lagi.

In [ ]:
path_baseline, losses_baseline = train("baseline")

In [ ]:
images_baseline = sample(path_baseline)

In [ ]:
# Uji eksekusi baseline
run_baseline = _resolve_run("baseline")
assert path_baseline == run_baseline.checkpoint_path and path_baseline.exists()
assert len(losses_baseline) == run_baseline.epochs
assert all(math.isfinite(v) for v in losses_baseline)
assert run_baseline.loss_json.exists() and run_baseline.loss_png.exists()
assert images_baseline.shape[0] == config.NUM_SAMPLE_IMAGES
# Tren loss hanya dicetak: di smoke test datanya acak, jadi loss belum tentu turun
print(f"loss epoch 1 = {losses_baseline[0]:.4f}  ->  epoch {len(losses_baseline)} = {losses_baseline[-1]:.4f}")
print("eksekusi baseline OK")

## 13. Eksperimen dan hasil (Teman B)
Sisa konfigurasi dilatih satu per satu, lalu loss-nya dibandingkan dalam dua grafik: kelompok learning rate dan kelompok T. Ingat, loss rendah belum tentu digitnya bagus, jadi grid akhir tiap eksperimen juga perlu dilihat.

In [ ]:
snapshot_13 = copy.deepcopy(vars(config))  # dipakai sel uji di akhir bagian ini
# Baseline sudah dilatih di bagian 12; sisanya dilatih di sini
experiment_checkpoints = {"baseline": path_baseline}
for name in EXPERIMENTS:
    if name != "baseline":
        experiment_checkpoints[name], _ = train(name)

In [ ]:
def _load_loss_records(names):
    """Baca catatan loss JSON tiap konfigurasi; mode smoke ikut config.SMOKE_TEST."""
    return {name: json.loads(_resolve_run(name).loss_json.read_text()) for name in names}

In [ ]:
# Grafik perbandingan: tiap kelompok berisi baseline + eksperimen yang mengubah variabel itu
records = _load_loss_records(EXPERIMENTS)
suffix = _resolve_run("baseline").suffix
for variable in ("LEARNING_RATE", "T"):
    group = ["baseline"] + [n for n, o in EXPERIMENTS.items() if variable in o]
    series = {n: records[n]["losses"] for n in group}
    plot_loss(series, config.LOSS_CURVES_DIR / f"perbandingan_{variable.lower()}{suffix}.png")

In [ ]:
# Grid akhir tiap eksperimen dari checkpoint-nya (baseline sudah di bagian 12)
for name, path in experiment_checkpoints.items():
    if name != "baseline":
        sample(path)

In [ ]:
# Tabel hasil ke results/experiments.md (smoke: experiments_smoke.md)
rows = ["| Konfigurasi | Learning rate | T | Epoch | Loss akhir | Loss minimum | Rata-rata detik/epoch | Catatan visual |",
        "|---|---|---|---|---|---|---|---|"]
for name, rec in records.items():
    secs, losses = rec["epoch_seconds"], rec["losses"]
    rows.append(f"| {name} | {rec['learning_rate']:g} | {rec['T']} | {len(losses)} | {losses[-1]:.4f} | "
                f"{min(losses):.4f} | {sum(secs) / len(secs):.1f} |  |")
header = ["# Hasil eksperimen", "",
          "Loss = rata-rata MSE noise per epoch. Kolom catatan visual diisi manual dari grid akhir.", ""]
md_path = config.RESULTS_DIR / f"experiments{suffix}.md"
md_path.parent.mkdir(parents=True, exist_ok=True)
md_path.write_text("\n".join(header + rows) + "\n")
print("\n".join(rows))

In [ ]:
# Uji eksperimen: timpaan benar-benar dipakai dan tersimpan, config tetap utuh
for name, overrides in EXPERIMENTS.items():
    run = _resolve_run(name)
    ckpt = torch.load(run.checkpoint_path, map_location=get_device(), weights_only=True)
    lr = ckpt["optimizer_state_dict"]["param_groups"][0]["lr"]
    assert ckpt["T"] == overrides.get("T", config.T), f"{name}: T di checkpoint salah"
    assert lr == overrides.get("LEARNING_RATE", config.LEARNING_RATE), f"{name}: LR salah"
    assert len(records[name]["losses"]) == run.epochs, f"{name}: jumlah epoch salah"
    print(f"{name:<9} T={ckpt['T']:<4} lr={lr:g}  OK")
assert len(rows) - 2 == len(EXPERIMENTS), "tabel harus punya satu baris per eksperimen"
assert vars(config) == snapshot_13, "config tidak boleh berubah"
print("eksperimen OK")